# Construcción de la matriz de diversidad
Este notebook reúne los archivos de métricas de diversidad generados a partir de los resultados obtenidos mediante el clustering de secuencias con Change-O.

La matriz construida corresponde a la Matriz A, que representa las métricas de diversidad calculadas sobre los clones inferidos mediante el pipeline de análisis.

## Librerías utilizadas

* **glob**: permite buscar automáticamente todos los archivos `.tsv` dentro de un directorio.
* **polars**: se utiliza para leer, manipular y guardar tablas de datos de forma eficiente.

---

## Búsqueda de archivos

```python
glob.glob(...)
```

* Busca todos los archivos con extensión `.tsv` ubicados en la carpeta `diversity_metrics`.
* Cada archivo corresponde a las métricas calculadas para una simulación.

---

## Verificación de archivos

```python
if not files:
```

* Comprueba que se hayan encontrado archivos.
* Si la carpeta está vacía, se genera un error (`FileNotFoundError`) para evitar continuar con el análisis.
---

## Lectura y unión de datos

```python
pl.concat(...)
```

* Lee cada archivo TSV mediante `pl.read_csv()`.
* Une todos los archivos en una única tabla utilizando `vertical_relaxed`, que concatena las filas permitiendo pequeñas diferencias entre columnas si existen.

---

## Ordenamiento de la matriz

```python
df_diversity.sort(...)
```

La matriz se ordena según:

* **condition**: escenario de simulación.
* **size**: número de secuencias.

Esto garantiza que los datos mantengan un orden consistente para los análisis y visualizaciones posteriores.

---

## Verificación del resultado
```python
print(...)
```

Se muestran las columnas:

* **sample_id**
* **condition**
* **size**

para comprobar que el orden de la matriz sea el esperado.

---

## Guardado de la matriz

```python
write_csv(...)
```

La matriz final se guarda como un único archivo TSV (`diversity_matrix_A.tsv`), el cual será utilizado en las etapas posteriores del análisis y la generación de gráficos.


In [1]:
import glob
import polars as pl

# Buscar todos los TSV
files = glob.glob(
    "/Users/catg/Desktop/SOFIAC/Gitsofia/tesisbioinf-sofia/results/diversity_metrics/downsampling/*.tsv"
)

# Verificar que existan archivos
if not files:
    raise FileNotFoundError(
        "No se encontraron archivos TSV en diversity_metrics"
    )

# Leer y unir
df_diversity = pl.concat(
    [pl.read_csv(f, separator="\t") for f in files],
    how="vertical_relaxed"
)

# Ordenar
df_diversity = df_diversity.sort(
    by=["condition", "size"],
    descending=[False, False]
)

# Verificar orden
print(
    df_diversity.select(
        ["sample_id", "condition", "size"]
    )
)

# Guardar matriz ordenada
df_diversity.write_csv(
    "/Users/catg/Desktop/SOFIAC/Gitsofia/tesisbioinf-sofia/results/diversity_matrix/diversity_matrix_DOWN.tsv",
    separator="\t"
)

shape: (55, 3)
┌────────────┬───────────┬────────┐
│ sample_id  ┆ condition ┆ size   │
│ ---        ┆ ---       ┆ ---    │
│ str        ┆ str       ┆ i64    │
╞════════════╪═══════════╪════════╡
│ A100seq    ┆ A         ┆ 100    │
│ A200seq    ┆ A         ┆ 200    │
│ A400seq    ┆ A         ┆ 400    │
│ A800seq    ┆ A         ┆ 800    │
│ A1600seq   ┆ A         ┆ 1600   │
│ …          ┆ …         ┆ …      │
│ E6400seq   ┆ E         ┆ 6400   │
│ E12800seq  ┆ E         ┆ 12800  │
│ E25600seq  ┆ E         ┆ 25600  │
│ E51200seq  ┆ E         ┆ 51200  │
│ E102400seq ┆ E         ┆ 102400 │
└────────────┴───────────┴────────┘


# Construcción de la matriz de diversidad A-ChangeO y downsampling

Este notebook reúne en una única matriz los archivos de métricas de diversidad correspondientes a los resultados obtenidos mediante el clustering de secuencias con **Change-O** y a los datos de **downsampling**.

La matriz integrada permite trabajar con ambos conjuntos de datos bajo una estructura común y facilita la posterior **estandarización de las métricas mediante Z-scores**, manteniendo identificada la procedencia de cada registro mediante la columna `source`.

## Librerías utilizadas

* **`glob`**: permite buscar automáticamente los archivos `.tsv` dentro de los directorios especificados.
* **`polars`**: se utiliza para leer, unir, ordenar y guardar las tablas de datos de manera eficiente.

---

## Búsqueda de archivos

Se identifican por separado los archivos `.tsv` correspondientes a:

* **A-ChangeO**: archivos almacenados en `diversity_metrics/A/`.
* **DOWNsampling**: archivos almacenados en `diversity_metrics/downsampling/`.

Esto permite conservar la procedencia de cada conjunto de datos al momento de integrarlos.

---

## Lectura y clasificación de los datos

Cada conjunto de archivos se lee mediante `pl.read_csv()` y se incorpora una columna denominada **`source`**, que identifica su origen:

* `A-ChangeO`: métricas calculadas a partir de los clones inferidos mediante Change-O.
* `DOWNsampling`: métricas calculadas a partir de los repertorios obtenidos mediante el procedimiento de downsampling.

---

## Unión de las matrices

Los archivos de ambos conjuntos se integran mediante:

```python
pl.concat(...)
```

La concatenación se realiza de forma vertical, reuniendo las filas de **A-ChangeO** y **DOWNsampling** en una única matriz.

La columna `source` permite distinguir posteriormente ambos tipos de datos sin perder su procedencia.

---

## Ordenamiento de la matriz

La matriz integrada se ordena según:

* **`condition`**: escenario de simulación.
* **`size`**: número de secuencias analizadas.

Esto permite mantener un orden consistente entre los diferentes escenarios y profundidades de secuenciación.

---

## Verificación del resultado

Se revisan las columnas principales:

* **`sample_id`**
* **`condition`**
* **`size`**
* **`source`**

La columna `source` permite comprobar que cada registro corresponda al conjunto de datos correcto.

---

## Utilidad para la estandarización mediante Z-scores

La matriz integrada se utiliza como base para la **estandarización de las métricas de diversidad mediante Z-scores**.

La integración permite que las métricas sean organizadas bajo una estructura común antes de su transformación, facilitando la comparación entre los diferentes escenarios y profundidades de secuenciación.

La identificación mediante `source` permite además distinguir posteriormente los resultados correspondientes a **A-ChangeO** y **DOWNsampling** en las visualizaciones y análisis posteriores.

---

## Guardado de la matriz

La matriz final se guarda como un único archivo TSV:

```python
diversity_matrix_A_DOWN.tsv
```

Este archivo será utilizado como entrada para las etapas posteriores de **estandarización mediante Z-scores, comparación de métricas de diversidad y generación de mapas de calor**.


In [3]:
import glob
import polars as pl

base = "/Users/catg/Desktop/SOFIAC/Gitsofia/tesisbioinf-sofia/results"

# Buscar archivos
files_A = glob.glob(f"{base}/diversity_metrics/A/*.tsv")
files_DOWN = glob.glob(f"{base}/diversity_metrics/downsampling/*.tsv")

# Leer archivos A-ChangeO
df_A = (
    pl.concat(
        [pl.read_csv(f, separator="\t") for f in files_A],
        how="vertical_relaxed"
    )
    .with_columns(
        pl.lit("A-ChangeO").alias("source")
    )
)

# Leer archivos de downsampling
df_DOWN = (
    pl.concat(
        [pl.read_csv(f, separator="\t") for f in files_DOWN],
        how="vertical_relaxed"
    )
    .with_columns(
        pl.lit("Downsampling").alias("source")
    )
)

# Unir
df_diversity = pl.concat(
    [df_A, df_DOWN],
    how="vertical_relaxed"
)

# Ordenar
df_diversity = df_diversity.sort(
    by=["condition", "size"],
    descending=[False, False]
)

# Verificar
print(
    df_diversity.select(
        ["sample_id", "condition", "size", "source"]
    )
)

# Guardar
df_diversity.write_csv(
    f"{base}/diversity_matrix/diversity_matrix_A_DOWN.tsv",
    separator="\t"
)

shape: (110, 4)
┌────────────┬───────────┬────────┬──────────────┐
│ sample_id  ┆ condition ┆ size   ┆ source       │
│ ---        ┆ ---       ┆ ---    ┆ ---          │
│ str        ┆ str       ┆ i64    ┆ str          │
╞════════════╪═══════════╪════════╪══════════════╡
│ A100seq    ┆ A         ┆ 100    ┆ Downsampling │
│ A100seq    ┆ A         ┆ 100    ┆ A-ChangeO    │
│ A200seq    ┆ A         ┆ 200    ┆ Downsampling │
│ A200seq    ┆ A         ┆ 200    ┆ A-ChangeO    │
│ A400seq    ┆ A         ┆ 400    ┆ Downsampling │
│ …          ┆ …         ┆ …      ┆ …            │
│ E25600seq  ┆ E         ┆ 25600  ┆ A-ChangeO    │
│ E51200seq  ┆ E         ┆ 51200  ┆ Downsampling │
│ E51200seq  ┆ E         ┆ 51200  ┆ A-ChangeO    │
│ E102400seq ┆ E         ┆ 102400 ┆ Downsampling │
│ E102400seq ┆ E         ┆ 102400 ┆ A-ChangeO    │
└────────────┴───────────┴────────┴──────────────┘
